In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

os.chdir(r'C:\Users\hdped\Desktop\ANTAQ_Projeto - Copia')

CORES = {
    'COSCO': '#003087', 'MAERSK': '#0099CC', 'CMA CGM': '#D62728',
    'MSC': '#F5C518', 'EVERGREEN': '#2CA02C', 'Hapag-Lloyd': '#F26722',
    'ONE': '#E377C2', 'ZIM': '#17BECF',
}
ORDEM_PRESSAO = ['Alta (< 2010)', 'Moderada (2010-2015)', 'Baixa (> 2015)', 'Indeterminado']
CORES_PRESSAO = {
    'Alta (< 2010)':        '#c00000',
    'Moderada (2010-2015)': '#f26722',
    'Baixa (> 2015)':       '#2CA02C',
    'Indeterminado':        '#aaaaaa',
}
plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False, 'figure.dpi': 120,
})
print('Setup OK')

In [ ]:
df = pd.read_csv('data/02_Operacoes/Vessels_Master_Enriched.csv',
                  encoding='utf-8-sig', low_memory=False)
df.columns = df.columns.str.strip()

n_total  = len(df)
n_null   = df['year_built'].isna().sum()
pct_null = n_null / n_total * 100
teu_null = df[df['year_built'].isna()]['CAPACIDADE (TEU)'].sum()
teu_tot  = df['CAPACIDADE (TEU)'].sum()

print('=== DIAGNÓSTICO year_built ===')
print(f'Total navios        : {n_total:,}')
print(f'year_built nulos    : {n_null} ({pct_null:.1f}%)')
print(f'TEU sem year_built  : {teu_null:,} ({teu_null/teu_tot*100:.1f}% do total)')
print()
print('AVISO: DWT 100% nulo — não usado. Análise baseada em year_built como proxy.')
print('       Esta análise NÃO calcula rating CII real.')

miss_teu = (df[df['year_built'].isna()]
            .groupby('SHIPPING LINE')['CAPACIDADE (TEU)'].agg(['sum','count'])
            .rename(columns={'sum':'teu_indet','count':'navios_indet'}))
total_c  = df.groupby('SHIPPING LINE')['CAPACIDADE (TEU)'].sum().rename('teu_total')
miss_teu = miss_teu.join(total_c)
miss_teu['pct_indet'] = (miss_teu['teu_indet'] / miss_teu['teu_total'] * 100).round(1)
print()
print('TEU indeterminado por carrier:')
print(miss_teu.sort_values('pct_indet', ascending=False))

In [ ]:
def pressao(y):
    if pd.isna(y): return 'Indeterminado'
    elif y < 2010: return 'Alta (< 2010)'
    elif y <= 2015: return 'Moderada (2010-2015)'
    else: return 'Baixa (> 2015)'

df['pressao_renovacao'] = df['year_built'].apply(pressao)

q1 = (df.groupby(['SHIPPING LINE', 'pressao_renovacao'])['CAPACIDADE (TEU)']
        .sum().reset_index(name='teu'))
q1_piv = q1.pivot_table(index='SHIPPING LINE', columns='pressao_renovacao',
                         values='teu', fill_value=0)
for col in ORDEM_PRESSAO:
    if col not in q1_piv.columns: q1_piv[col] = 0
q1_piv = q1_piv[ORDEM_PRESSAO]
q1_piv['total'] = q1_piv.sum(axis=1)
for col in ORDEM_PRESSAO:
    q1_piv[f'pct_{col}'] = (q1_piv[col] / q1_piv['total'] * 100).round(1)
q1_piv = q1_piv.sort_values('pct_Alta (< 2010)', ascending=False)

q1_piv['teu_conhecido'] = q1_piv['Alta (< 2010)'] + q1_piv['Moderada (2010-2015)'] + q1_piv['Baixa (> 2015)']
q1_piv['pct_alta_det'] = (q1_piv['Alta (< 2010)'] / q1_piv['teu_conhecido'] * 100).round(1)

print('=== Ranking % TEU Pressão Alta (sobre navios com year_built conhecido) ===')
print(q1_piv[['pct_alta_det','pct_Alta (< 2010)','pct_Moderada (2010-2015)','pct_Baixa (> 2015)','pct_Indeterminado']]
      .rename(columns={
          'pct_alta_det': '% Alta (det.)',
          'pct_Alta (< 2010)': '% Alta (total)',
          'pct_Moderada (2010-2015)': '% Moderada',
          'pct_Baixa (> 2015)': '% Baixa',
          'pct_Indeterminado': '% Indet.'
      }).to_string())

In [ ]:
carriers_order = q1_piv.index.tolist()
x = np.arange(len(carriers_order))

fig, ax = plt.subplots(figsize=(13, 6))
fig.suptitle('Pressão de Renovação de Frota por Carrier — % TEU por Faixa de year_built\n'
             '(Proxy IMO CII — não é cálculo de rating CII real)',
             fontsize=12, fontweight='bold')

bottom = np.zeros(len(carriers_order))
for pressao_cat in ORDEM_PRESSAO:
    vals = q1_piv[f'pct_{pressao_cat}'].values
    bars = ax.bar(x, vals, bottom=bottom, color=CORES_PRESSAO[pressao_cat],
                  alpha=0.88, label=pressao_cat, width=0.6)
    for i, (bar, val) in enumerate(zip(bars, vals)):
        if val > 7:
            ax.text(bar.get_x() + bar.get_width()/2, bottom[i] + val/2,
                    f'{val:.0f}%', ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
    bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(carriers_order, rotation=0)
ax.set_ylabel('% Capacidade TEU')
ax.set_ylim(0, 105)
ax.legend(fontsize=9, loc='upper right', title='Pressão de Renovação')
ax.text(0.01, -0.12,
        f'Limitações: year_built é proxy — retrofits alteram CII real. '
        f'{pct_null:.0f}% dos navios sem year_built (Indeterminado). '
        'DWT 100% nulo — calado não avaliado.',
        transform=ax.transAxes, fontsize=7.5, color='#666', va='top')
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Ranking de Carriers — Exposição à Pressão de Renovação\n'
             '(% calculado sobre navios com year_built conhecido)',
             fontsize=12, fontweight='bold')

colors_rank = [CORES.get(c, '#888') for c in carriers_order]
x_r = np.arange(len(carriers_order))

bars1 = ax1.bar(x_r, q1_piv['pct_alta_det'].values, color=colors_rank, alpha=0.88, width=0.6)
ax1.set_xticks(x_r)
ax1.set_xticklabels(carriers_order, rotation=30, ha='right')
ax1.set_ylabel('% TEU em Pressão Alta')
ax1.set_title('Ranking % TEU Pressão Alta\n(year_built < 2010 / sobre navios conhecidos)')
ax1.axhline(30, color='gray', linestyle='--', alpha=0.4, label='30%')
ax1.legend(fontsize=8)
for bar, val in zip(bars1, q1_piv['pct_alta_det'].values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

bottom2 = np.zeros(len(carriers_order))
for pressao_cat in ORDEM_PRESSAO:
    vals2 = q1_piv[pressao_cat].values
    ax2.bar(x_r, vals2, bottom=bottom2, color=CORES_PRESSAO[pressao_cat],
            alpha=0.88, label=pressao_cat, width=0.6)
    bottom2 += vals2
ax2.set_xticks(x_r)
ax2.set_xticklabels(carriers_order, rotation=30, ha='right')
ax2.set_ylabel('TEU')
ax2.set_title('Volume TEU por Faixa de Pressão')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,p: f'{v/1e6:.1f}M'))
ax2.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
SEG_ORDER = ['Feeder Max (1k-3k TEU)', 'Sub-Panamax (3k-8k TEU)',
             'Post-Panamax (8k-12k TEU)', 'New Panamax (12k-18k TEU)',
             'Ultra Large (> 18k TEU)']
SEG_LABELS = ['Feeder Max\n(1k-3k)', 'Sub-Panamax\n(3k-8k)',
              'Post-Panamax\n(8k-12k)', 'New Panamax\n(12k-18k)', 'Ultra Large\n(>18k)']

q3 = (df.groupby(['vessel_segment', 'pressao_renovacao'])['CAPACIDADE (TEU)']
        .sum().reset_index(name='teu'))
q3_piv = q3.pivot_table(index='vessel_segment', columns='pressao_renovacao',
                          values='teu', fill_value=0)
for col in ORDEM_PRESSAO:
    if col not in q3_piv.columns: q3_piv[col] = 0
q3_piv = q3_piv[ORDEM_PRESSAO]
q3_piv['total'] = q3_piv.sum(axis=1)
q3_piv['teu_conhecido'] = q3_piv['Alta (< 2010)'] + q3_piv['Moderada (2010-2015)'] + q3_piv['Baixa (> 2015)']
q3_piv['pct_alta_det'] = (q3_piv['Alta (< 2010)'] / q3_piv['teu_conhecido'] * 100).round(1)
q3_piv = q3_piv.reindex([s for s in SEG_ORDER if s in q3_piv.index])

segs = q3_piv.index.tolist()
seg_labels_actual = [SEG_LABELS[SEG_ORDER.index(s)] if s in SEG_ORDER else s for s in segs]
x_s = np.arange(len(segs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Pressão de Renovação por Vessel Segment',
             fontsize=12, fontweight='bold')

bottom3 = np.zeros(len(segs))
for pressao_cat in ORDEM_PRESSAO:
    vals3 = q3_piv[pressao_cat].values / q3_piv['total'].values * 100
    bars3 = ax1.bar(x_s, vals3, bottom=bottom3, color=CORES_PRESSAO[pressao_cat],
                    alpha=0.88, label=pressao_cat, width=0.6)
    for i, (bar, val) in enumerate(zip(bars3, vals3)):
        if val > 8:
            ax1.text(bar.get_x() + bar.get_width()/2, bottom3[i] + val/2,
                     f'{val:.0f}%', ha='center', va='center',
                     fontsize=8.5, color='white', fontweight='bold')
    bottom3 += vals3
ax1.set_xticks(x_s); ax1.set_xticklabels(seg_labels_actual, fontsize=9)
ax1.set_ylabel('% Capacidade TEU'); ax1.set_ylim(0, 105)
ax1.set_title('Distribuição por Faixa de Pressão')
ax1.legend(fontsize=8, loc='upper right')

bar_colors2 = ['#c00000' if v > 60 else '#f26722' if v > 30 else '#2CA02C'
               for v in q3_piv['pct_alta_det'].values]
bars4 = ax2.bar(x_s, q3_piv['pct_alta_det'].values, color=bar_colors2, alpha=0.88, width=0.6)
ax2.set_xticks(x_s); ax2.set_xticklabels(seg_labels_actual, fontsize=9)
ax2.set_ylabel('% TEU em Pressão Alta')
ax2.set_title('% TEU Pressão Alta por Segmento\n(sobre navios com year_built conhecido)')
ax2.axhline(30, color='gray', linestyle='--', alpha=0.4)
for bar, val in zip(bars4, q3_piv['pct_alta_det'].values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.0f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
q1_piv.reset_index().to_csv('outputs/processed_data/cii_q1_pressao_carrier.csv',
                             index=False, encoding='utf-8-sig')
q3_piv.reset_index().to_csv('outputs/processed_data/cii_q3_pressao_segmento.csv',
                              index=False, encoding='utf-8-sig')

print('CSVs exportados para outputs/processed_data/')
print()
print('=' * 55)
print('SUMÁRIO')
print('=' * 55)
print(f'Carrier maior pressão alta : COSCO  54% do TEU determinado')
print(f'Carrier menor pressão alta : ONE    16% do TEU determinado')
print(f'Segmento crítico           : Sub-Panamax 90% em pressão alta')
print(f'Ultra Large                : 0% em pressão alta (todos pós-2015)')
print()
print('LIMITAÇÕES:')
print(f'  1. year_built é proxy — retrofits alteram CII real')
print(f'  2. {pct_null:.0f}% dos navios sem year_built — COSCO mais afectado (26% TEU indeterminado)')
print(f'  3. DWT 100% nulo — calado não avaliado')
print(f'  4. Não é cálculo de rating CII real')